### Import

In [3]:
import torch
import torchvision
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Torch: 2.8.0+cu126
TorchVision: 0.23.0+cu126
Using device: cuda


In [2]:
import unsloth
print(unsloth.__version__)

[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


C:\Users\yujie.lim\AppData\Roaming\Python\Python310\site-packages\unsloth_zoo\_vendored\fla\utils\_device.py:100: UserWarning: Triton is not supported on current platform, roll back to CPU.
  _cpu_device_warning()
C:\Users\yujie.lim\AppData\Roaming\Python\Python310\site-packages\unsloth_zoo\_vendored\fla\utils\_device.py:165: UserWarning: Triton is not supported on current platform, roll back to CPU.
  _cpu_device_warning()
C:\Users\yujie.lim\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0902 16:45:45.785000 25352 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
2026.8.22


In [4]:
import json
from pathlib import Path
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    model_name="unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.for_training(model)
print(f"Model ready on {device}")

==((====))==  Unsloth 2026.8.22: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 729/729 [00:01<00:00, 420.90it/s]


Model ready on cuda


In [5]:
fourbit_models = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 473/473 [00:02<00:00, 160.19it/s]


### Train-test-split

In [ ]:
from datasets import load_dataset
dataset = load_dataset("unsloth/LaTeX_OCR", split = "train")

Generating test split: 100%|██████████| 7632/7632 [00:00<00:00, 144866.80 examples/s]


In [21]:
from datasets import Dataset

data = {
    "image": [
        r"images\myKad0.jpg",
        r"images\myKad1.jpg",
        r"images\myKad2.jpg"
    ],
    "text": [
        """{
          "Id-no": "030612-14-0440",
          "Name": "Yong Jia Yi",
          "Address1": "17 JALAN 1/2A",
          "Address2": "BANDAR DAMAI PERDANA",
          "Address3": "CHERAS",
          "Postal_Code": "56000 KUALA LUMPUR",
          "State": "W.PERSEKUTAN(KL)",
          "Gender": "PEREMPUAN"
        }""",

        """{
          "Id-no": "961024-23-5131",
          "Name": "SRISEENIVASAN A/L P SUBRAMANIAM",
          "Address1": "NO 1",
          "Address2": "JALAN LANJUT 9",
          "Address3": "TAMAN DESA CEMERLANG",
          "Postal_code": "81800 ULU TIRAM",
          "State": "JOHOR",
          "Gender": "LELAKI"
        }""",

        """{
          "Id-no": "040825-13-0755",
          "Name": "LUZMAN AL THAQIF BIN NARZARUDDIN",
          "Address1": "NO 1",
          "Address2": "FLAT SUNGAI RAJANG",
          "Address3": "JALAN REPOK",
          "Postal_Code": "96100",
          "State": "SARAWAK",
          "Gender": "LELAKI"
        }"""
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['image', 'text'],
    num_rows: 3
})


In [22]:

# Split it - 80/20 is standard, adjust if needed
train_test = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test['train']
test_dataset = train_test['test']

# For QLoRA fine-tuning with Qwen-VL, you'll need a collate function
def collate_fn(batch):
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    # Process based on your model's processor
    return {'images': images, 'texts': texts}

# Then pass to DataLoader
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, collate_fn=collate_fn)

In [23]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3_5VisionModel; please override in the subclass.. Falling back to pre-forward hook.


In [3]:
for i in range(100):
    with open("something.txt", "a") as f:
        f.write(f'r"images\myKad{i}.jpg",\n')
